# Day 26: Observability — Metrics, Tracing & Dashboards
> *Inference Engineering* — Chapter 7.4.3 | Philip Kiely, Baseten Books 2026

**Layer:** Tooling | **Prerequisite:** Day 25 (Benchmarking)


## Concept Overview

Observability for inference systems requires three pillars: metrics (aggregated time-series), traces (per-request latency breakdown), and logs. Key metrics: TTFT, TPOT, token throughput, GPU utilization, KV cache utilization, queue depth, and error rate. Distributed tracing tracks a single request through the entire stack: from API gateway → scheduler → prefill GPU → decode GPU → response. Prometheus collects metrics; Grafana visualizes them. OpenTelemetry provides a standard SDK for trace instrumentation.


In [ ]:
import time, random
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional, Dict, List
from collections import defaultdict

print('Observability stack simulation')


## 1. Prometheus Metrics

The canonical inference server metrics exposed in Prometheus format.


In [ ]:
class InferenceMetrics:
    """Simulates a Prometheus metrics registry for an inference server."""
    def __init__(self):
        self.request_count = 0
        self.error_count = 0
        self.ttft_histogram = []
        self.tpot_histogram = []
        self.token_count = 0
        self.gpu_util = []
        self.kv_cache_util = []
        self.queue_depth = []

    def record_request(self, ttft_ms, tpot_ms, num_output_tokens,
                       success=True, gpu_util=0.8, kv_util=0.6, queue=2):
        self.request_count += 1
        if not success:
            self.error_count += 1
        self.ttft_histogram.append(ttft_ms)
        self.tpot_histogram.append(tpot_ms)
        self.token_count += num_output_tokens
        self.gpu_util.append(gpu_util)
        self.kv_cache_util.append(kv_util)
        self.queue_depth.append(queue)

    def prometheus_export(self):
        lines = [
            f'# HELP inference_requests_total Total inference requests',
            f'inference_requests_total {self.request_count}',
            f'# HELP inference_errors_total Total inference errors',
            f'inference_errors_total {self.error_count}',
            f'# HELP inference_ttft_milliseconds P50/P95/P99 TTFT',
        ]
        if self.ttft_histogram:
            for p in [50, 95, 99]:
                v = np.percentile(self.ttft_histogram, p)
                lines.append(f'inference_ttft_milliseconds{{quantile="{p/100}"}} {v:.2f}')
        lines += [
            f'inference_tokens_total {self.token_count}',
            f'inference_gpu_utilization {np.mean(self.gpu_util):.3f}',
            f'inference_kv_cache_utilization {np.mean(self.kv_cache_util):.3f}',
            f'inference_queue_depth {self.queue_depth[-1] if self.queue_depth else 0}',
        ]
        return '\n'.join(lines)

# Simulate traffic
metrics = InferenceMetrics()
np.random.seed(42)
for i in range(500):
    metrics.record_request(
        ttft_ms=np.random.lognormal(4.0, 0.3),
        tpot_ms=np.random.lognormal(3.0, 0.2),
        num_output_tokens=np.random.randint(50, 500),
        success=np.random.random() > 0.005,
        gpu_util=np.clip(np.random.normal(0.78, 0.05), 0, 1),
        kv_util=np.clip(np.random.normal(0.65, 0.1), 0, 1),
        queue=np.random.poisson(3),
    )

print('/metrics endpoint output (Prometheus format):')
print(metrics.prometheus_export())


## 2. Distributed Tracing

Each request gets a trace with child spans for each processing stage. Trace data reveals where time is spent: is TTFT dominated by queue wait, network, or actual prefill?


In [ ]:
import uuid

@dataclass
class Span:
    trace_id: str
    span_id: str
    parent_id: Optional[str]
    operation: str
    start_ms: float
    end_ms: float
    tags: Dict = field(default_factory=dict)

    @property
    def duration_ms(self):
        return self.end_ms - self.start_ms

class Tracer:
    def __init__(self):
        self.traces = defaultdict(list)

    def trace_request(self, prompt_len, output_len):
        trace_id = str(uuid.uuid4())[:8]
        t = 0
        spans = []

        def add_span(op, duration, parent_id=None, **tags):
            nonlocal t
            span = Span(trace_id, str(uuid.uuid4())[:8], parent_id, op, t, t+duration, tags)
            spans.append(span)
            t += duration
            return span.span_id

        # Request lifecycle
        root_id = add_span('inference_request', 0)
        add_span('network_ingress', np.random.normal(2, 0.5), root_id)
        add_span('tokenization', np.random.normal(1, 0.2), root_id, prompt_len=prompt_len)
        add_span('queue_wait', np.random.exponential(5), root_id)
        add_span('prefill', np.random.normal(prompt_len * 0.1, 5), root_id,
                prompt_len=prompt_len, batch_size=4)
        decode_id = add_span('decode', output_len * np.random.normal(20, 2), root_id,
                           output_tokens=output_len)
        add_span('detokenization', np.random.normal(0.5, 0.1), root_id)
        add_span('network_egress', np.random.normal(1, 0.3), root_id)

        self.traces[trace_id] = spans
        return trace_id, spans

tracer = Tracer()
tid, spans = tracer.trace_request(prompt_len=256, output_len=100)

print(f'Trace {tid}:')
print(f'{"Operation":<20} {"Start (ms)":>12} {"Duration (ms)":>15}')
for span in spans:
    print(f'{span.operation:<20} {span.start_ms:>12.1f} {span.duration_ms:>15.1f}')
print(f'Total: {spans[-1].end_ms:.1f} ms')


## 3. Grafana Dashboard Layout

Standard inference dashboard panels and their alerting thresholds.


In [ ]:
# Simulate and visualize key dashboard panels
np.random.seed(42)
T = 60  # minutes
time_axis = np.arange(T)

# Simulate metrics over time
ttft_p99 = np.random.lognormal(4.5, 0.2, T) * (1 + 0.3 * np.sin(time_axis/10))
throughput = 200 + 100 * np.sin(time_axis/15) + np.random.normal(0, 10, T)
kv_util = np.clip(0.5 + 0.2 * np.sin(time_axis/12) + np.random.normal(0, 0.05, T), 0, 1)
queue_depth = np.maximum(0, np.random.poisson(3, T) * (1 + np.sin(time_axis/8)))
error_rate = np.clip(np.random.exponential(0.001, T), 0, 0.05)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for ax, data, title, ylabel, threshold, threshold_label in [
    (axes[0,0], ttft_p99, 'TTFT P99 (ms)', 'ms', 500, 'SLO 500ms'),
    (axes[0,1], throughput, 'Token Throughput', 'tokens/s', None, None),
    (axes[0,2], kv_util*100, 'KV Cache Utilization', '%', 90, 'Alert 90%'),
    (axes[1,0], queue_depth, 'Queue Depth', 'requests', 10, 'Alert 10'),
    (axes[1,1], error_rate*100, 'Error Rate', '%', 1, 'Alert 1%'),
]:
    ax.plot(time_axis, data)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('Time (min)')
    if threshold:
        ax.axhline(threshold, color='red', linestyle='--', alpha=0.7, label=threshold_label)
        ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# GPU utilization heatmap
ax = axes[1,2]
gpu_heatmap = np.random.beta(8, 2, (4, T))  # 4 GPUs over time
im = ax.imshow(gpu_heatmap, aspect='auto', vmin=0, vmax=1, cmap='RdYlGn')
plt.colorbar(im, ax=ax)
ax.set_title('GPU Utilization (4 workers)')
ax.set_xlabel('Time (min)')
ax.set_yticks(range(4))
ax.set_yticklabels([f'GPU {i}' for i in range(4)])

plt.suptitle('Inference Service Grafana Dashboard', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('grafana_dashboard.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved grafana_dashboard.png')


## Experiments: Try These

1. **Real Prometheus**: Add a Prometheus `/metrics` endpoint to a vLLM server. Configure Prometheus to scrape it and Grafana to visualize it.
2. **OpenTelemetry SDK**: Instrument a FastAPI wrapper with OpenTelemetry traces. Export to Jaeger (run with Docker) and view the waterfall trace.
3. **Alert rule**: Write a Prometheus alerting rule: fire PagerDuty if P95 TTFT > 500ms for 5 consecutive minutes.


## Key Takeaways

- The three observability pillars for inference: metrics (aggregate trends), traces (per-request breakdown), and logs (event context).
- Key Prometheus metrics: TTFT histogram, TPOT histogram, KV cache utilization, queue depth, GPU utilization, error rate.
- Distributed tracing reveals hidden latency: queue wait time often dominates TTFT during traffic spikes, masking actual prefill time improvements.
- KV cache utilization is the most actionable metric: if it's consistently >90%, you're OOMing and need more GPUs or shorter max sequences.

## References
- *Inference Engineering* Ch 7.4.3 — Philip Kiely, Baseten Books 2026
- Prometheus documentation
- OpenTelemetry specification
- Grafana dashboard best practices
